# Distributions: Understanding Data Through Probability
A distribution describes how probability is spread across values. It answers: "If I sample from this process, what values will I get and how often?"

We'll explore:
1. **Sampling**: How to generate random data from distributions
2. **Heavy tails**: Why rare events matter and break normal assumptions
3. **CDFs**: The fundamental tool for understanding any distribution

Core insight: Real-world data rarely follows the "nice" normal distribution. Understanding the tails is crucial for risk management, anomaly detection, and realistic modeling.

<details>

<summary>Probability Distributions vs Empirical Data</summary>
Probability distribution: The theoretical model (e.g., Normal with μ=100, σ=15).<br><br>
Empirical data: Samples from that (or an unknown) distribution. Your data is random; the distribution is fixed.
</details>

## The Problem: Modeling Daily Stock Returns
Stock market risk depends on understanding return distributions. Two portfolios with same average return can have vastly different risks if one has "heavy tails" (extreme swings).

Let's compare:
- **Normal distribution**: Used by classical finance (assumes symmetry, rare extremes)
- **Student-t distribution**: Heavy tails, better captures crashes/rallies
- **Empirical 2008 data**: What actually happened

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

np.random.seed(42)

# Parameters
mu = 0.0005       # Average daily return (~0.05%)
sigma = 0.015     # Daily volatility (~1.5%)
n_samples = 10000

# Generate samples from different distributions
normal_returns = np.random.normal(mu, sigma, n_samples)
t_returns = mu + sigma * np.random.standard_t(df=4, size=n_samples)  # df=4 gives heavy tails

# Real-world-ish data: mix of normal + occasional crashes
empirical_returns = np.random.normal(mu, sigma, n_samples)
# Add 10 crash days (outliers)
crash_indices = np.random.choice(n_samples, 10, replace=False)
empirical_returns[crash_indices] = np.random.uniform(-0.10, -0.05, 10)  # 5-10% down days

# Visualize
plt.figure(figsize=(15, 5))

# Histogram
plt.subplot(1, 3, 1)
plt.hist(normal_returns, bins=50, alpha=0.6, label='Normal', edgecolor='black', density=True)
plt.hist(t_returns, bins=50, alpha=0.6, label='Student-t (df=4)', edgecolor='black', density=True)
plt.hist(empirical_returns, bins=50, alpha=0.6, label='Empirical (w/ crashes)', edgecolor='black', density=True)
plt.xlabel('Daily Return')
plt.ylabel('Density')
plt.title('Distribution of Returns')
plt.legend()
plt.grid(alpha=0.3)

# Zoomed on left tail
plt.subplot(1, 3, 2)
bins_tail = np.linspace(-0.12, -0.02, 40)
plt.hist(normal_returns[normal_returns < -0.02], bins=bins_tail, alpha=0.6, label='Normal', edgecolor='black', density=True)
plt.hist(t_returns[t_returns < -0.02], bins=bins_tail, alpha=0.6, label='Student-t', edgecolor='black', density=True)
plt.hist(empirical_returns[empirical_returns < -0.02], bins=bins_tail, alpha=0.6, label='Empirical', edgecolor='black', density=True)
plt.xlabel('Daily Return')
plt.ylabel('Density')
plt.title('Left Tail: Where Risk Lives')
plt.legend()
plt.grid(alpha=0.3)

# Q-Q plot (Normal vs t)
plt.subplot(1, 3, 3)
stats.probplot(normal_returns, dist="norm", plot=plt)
plt.title('Q-Q Plot: Normal Distribution')
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Dataset: {n_samples} daily returns")
print(f"\nNormal:      μ={normal_returns.mean():.6f}, σ={normal_returns.std():.6f}")
print(f"Student-t:   μ={t_returns.mean():.6f}, σ={t_returns.std():.6f}")
print(f"Empirical:   μ={empirical_returns.mean():.6f}, σ={empirical_returns.std():.6f}")

## Sampling: Generating Data from Distributions
How do we sample from complex distributions? Three main approaches:

1. **Inverse Transform**: If CDF is invertible, sample $u \sim U[0,1]$, return $F^{-1}(u)$
2. **Rejection Sampling**: Generate candidates, keep those that fit (wasteful but simple)
3. **MCMC/Other**: For high-dim distributions (beyond scope here)

In [ ]:
# Implement inverse transform sampling for exponential distribution
def sample_exponential_inverse_transform(lam, n_samples):
    """
    Sample from Exponential(λ) using inverse transform.
    CDF: F(x) = 1 - exp(-λx)
    Inverse: F^{-1}(u) = -ln(1-u) / λ
    """
    u = np.random.uniform(0, 1, n_samples)
    return -np.log(1 - u) / lam

# Implement rejection sampling for a custom distribution
def sample_beta_rejection(a, b, n_samples):
    """
    Sample from Beta(a, b) using rejection sampling.
    Propose from Uniform[0,1], accept with probability proportional to Beta PDF.
    """
    samples = []
    # Max of Beta(a,b) PDF (for acceptance threshold)
    max_pdf = stats.beta.pdf(np.linspace(0, 1, 1000), a, b).max()
    
    while len(samples) < n_samples:
        # Propose from uniform
        x = np.random.uniform(0, 1)
        # Compute target PDF
        pdf_val = stats.beta.pdf(x, a, b)
        # Accept with probability pdf_val / max_pdf
        if np.random.uniform(0, 1) < pdf_val / max_pdf:
            samples.append(x)
    
    return np.array(samples)

# Generate samples
n_samples_demo = 5000

# Exponential using inverse transform
exp_samples = sample_exponential_inverse_transform(lam=1.0, n_samples=n_samples_demo)
exp_samples_scipy = np.random.exponential(scale=1.0, size=n_samples_demo)  # For comparison

# Beta using rejection
beta_samples = sample_beta_rejection(a=2, b=5, n_samples=n_samples_demo)
beta_samples_scipy = np.random.beta(2, 5, n_samples_demo)  # For comparison

# Visualize
plt.figure(figsize=(14, 5))

# Exponential
plt.subplot(1, 3, 1)
x_exp = np.linspace(0, 5, 200)
plt.hist(exp_samples, bins=50, alpha=0.6, density=True, label='Our inverse transform', edgecolor='black')
plt.plot(x_exp, stats.expon.pdf(x_exp, scale=1.0), 'r-', linewidth=2, label='True PDF')
plt.xlabel('Value')
plt.ylabel('Density')
plt.title('Exponential via Inverse Transform')
plt.legend()
plt.grid(alpha=0.3)

# Beta
plt.subplot(1, 3, 2)
x_beta = np.linspace(0, 1, 200)
plt.hist(beta_samples, bins=50, alpha=0.6, density=True, label='Our rejection sampling', edgecolor='black')
plt.plot(x_beta, stats.beta.pdf(x_beta, 2, 5), 'r-', linewidth=2, label='True PDF')
plt.xlabel('Value')
plt.ylabel('Density')
plt.title('Beta(2,5) via Rejection Sampling')
plt.legend()
plt.grid(alpha=0.3)

# Efficiency of rejection sampling
plt.subplot(1, 3, 3)
efficiency_data = []
for (a_val, b_val) in [(2, 2), (2, 5), (1, 1), (0.5, 0.5)]:
    # Estimate efficiency: samples kept / candidates tried
    pdf_vals = stats.beta.pdf(np.linspace(0, 1, 1000), a_val, b_val)
    max_pdf = pdf_vals.max()
    avg_pdf = pdf_vals.mean()
    efficiency = avg_pdf / max_pdf
    efficiency_data.append((f"Beta({a_val},{b_val})", efficiency))

names, effs = zip(*efficiency_data)
plt.bar(range(len(names)), effs, alpha=0.7, edgecolor='black')
plt.xticks(range(len(names)), names, rotation=15)
plt.ylabel('Efficiency (samples kept / candidates)')
plt.title('Rejection Sampling Efficiency')
plt.grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("Sampling methods:")
print(f"  Inverse Transform: Fast, exact. Requires invertible CDF.")
print(f"  Rejection Sampling: Slow (mean {1/efficiency_data[1][1]:.1f}x candidates for Beta(2,5)), but simple.")
print(f"  MCMC: For high-dimensional, no closed form.")

<details>

<summary>Why is sampling hard in high dimensions?</summary>
In 1D, rejection sampling works fine. In 100D, the unit hypercube is mostly empty—proposals rarely hit the target distribution. MCMC walks through the distribution instead of proposing blindly, making it feasible for high-d problems.
</details>

## Heavy Tails: When Rare Events Matter
**Heavy-tailed distributions** have more probability mass in the extremes than normal distributions. This matters for risk:
- A 5-sigma event is "essentially impossible" in normal (p ≈ 3e-7)
- In heavy-tailed, 5-sigma events happen regularly

**Skewness** and **Kurtosis** quantify tail behavior:

In [ ]:
# Generate different distributions
n_samples = 100000

# Normal (for reference)
normal = np.random.normal(0, 1, n_samples)

# Lognormal (skewed, semi-heavy-tailed)
lognormal = np.random.lognormal(mean=0, sigma=0.5, size=n_samples)

# Student-t (heavy-tailed)
t_dist = np.random.standard_t(df=3, size=n_samples)

# Cauchy (extremely heavy-tailed, no mean/variance!)
cauchy = np.random.standard_cauchy(n_samples)
# Clip for visualization (Cauchy has infinite range)
cauchy_clipped = np.clip(cauchy, -10, 10)

distributions = [
    ('Normal(0,1)', normal, 'Normal'),
    ('Lognormal', lognormal, 'Lognormal'),
    ('Student-t(df=3)', t_dist, 'Student-t'),
    ('Cauchy (clipped)', cauchy_clipped, 'Cauchy')
]

# Compute statistics
print(f"{'Distribution':<20} {'Skewness':<12} {'Kurtosis':<12} {'P(>3σ)':<12}")
print("=" * 56)

for name, data, label in distributions:
    skew = stats.skew(data)
    kurt = stats.kurtosis(data)
    # For cauchy, can't compute mean/std meaningfully, so skip P(>3σ)
    if label != 'Cauchy':
        mean, std = np.mean(data), np.std(data)
        # Kurtosis of normal = 0 (excess kurtosis); for t-dist > 0
        p_extreme = np.mean(np.abs(data - mean) > 3 * std)
        print(f"{name:<20} {skew:<12.4f} {kurt:<12.4f} {p_extreme:<12.4f}")
    else:
        print(f"{name:<20} {'N/A':<12} {'N/A':<12} {'N/A':<12}")

print("\nFor Normal: expected P(>3σ) ≈ 0.0027")
print("For Student-t(df=3): P(>3σ) is much higher!")
print("Cauchy: No mean or variance!")

## Visualizing Tails: Log Scale Plot
Heavy tails show up clearly on log-scale plots. A tail is heavier if the log-density decreases slower.

In [ ]:
# Create bins for right tail
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Right tail in log scale
ax = axes[0]
bin_edges = np.logspace(0, 1, 30)  # Logarithmic bins from 1 to 10

for name, data, label in distributions:
    if label != 'Cauchy':
        # Count values in each bin (right tail only)
        tail_data = data[data > 1]
        counts, _ = np.histogram(tail_data, bins=bin_edges)
        bin_centers = np.sqrt(bin_edges[:-1] * bin_edges[1:])  # Geometric mean
        ax.loglog(bin_centers, counts + 1e-3, 'o-', linewidth=2, label=label, markersize=6)

ax.set_xlabel('Value (log scale)')
ax.set_ylabel('Frequency (log scale)')
ax.set_title('Right Tail: Log-Log Plot')
ax.legend()
ax.grid(alpha=0.3, which='both')

# Distribution comparison (linear scale, focused on center)
ax = axes[1]
for name, data, label in distributions:
    if label != 'Cauchy':
        # Clip for visibility
        clipped = np.clip(data, -5, 5)
        ax.hist(clipped, bins=50, alpha=0.5, density=True, label=label, edgecolor='black')

ax.set_xlabel('Value')
ax.set_ylabel('Density')
ax.set_title('Center vs Tails (Linear Scale)')
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("On log-log plot:")
print("  Normal: Steep descent (exponential tail)")
print("  Student-t: Slower descent (power-law tail, heavier)")
print("  Cauchy: Very slow descent (extremely heavy)")

## CDFs: The Universal Tool
The **Cumulative Distribution Function (CDF)** $F(x) = P(X \leq x)$ is defined for ANY distribution. It's the fundamental bridge between data and probability.

**Key properties:**
- Monotone increasing: $F(x_1) \leq F(x_2)$ if $x_1 \leq x_2$
- Ranges [0, 1]: $F(-\infty) = 0$, $F(\infty) = 1$
- Quantile = inverse: $F^{-1}(0.95)$ is the 95th percentile

In [ ]:
# Plot CDFs
x_vals = np.linspace(-5, 5, 1000)

plt.figure(figsize=(14, 5))

# CDF comparison
plt.subplot(1, 2, 1)
for name, _, label in distributions[:-1]:  # Skip Cauchy
    if label == 'Normal(0,1)':
        cdf = stats.norm.cdf(x_vals)
    elif label == 'Lognormal':
        cdf = stats.lognorm.cdf(x_vals, s=0.5)
    elif label == 'Student-t':
        cdf = stats.t.cdf(x_vals, df=3)
    
    plt.plot(x_vals, cdf, linewidth=2, label=label)

plt.xlabel('Value')
plt.ylabel('CDF: P(X ≤ x)')
plt.title('CDFs: Notice tail behavior')
plt.legend()
plt.grid(alpha=0.3)

# Zoomed on upper tail
plt.subplot(1, 2, 2)
x_tail = np.linspace(2, 5, 1000)

for name, _, label in distributions[:-1]:
    if label == 'Normal(0,1)':
        cdf = stats.norm.cdf(x_tail)
    elif label == 'Lognormal':
        cdf = stats.lognorm.cdf(x_tail, s=0.5)
    elif label == 'Student-t':
        cdf = stats.t.cdf(x_tail, df=3)
    
    # Plot 1 - CDF (how much is left in tail)
    plt.plot(x_tail, 1 - cdf, linewidth=2, label=label)

plt.semilogy()
plt.xlabel('Value')
plt.ylabel('Tail probability: P(X > x) [log scale]')
plt.title('Upper Tails: Who decays slowest?')
plt.legend()
plt.grid(alpha=0.3, which='both')

plt.tight_layout()
plt.show()

print("Key insight: Tail probability (1-CDF) decays at different rates.")
print("Normal: Exponential decay (fastest)")
print("Student-t: Power-law decay (slower - heavy tails)")

## Empirical CDF (ECDF) and Goodness-of-Fit
Given data, the **empirical CDF** is $\hat{F}(x) = \frac{\#\{X_i \leq x\}}{n}$ (fraction of samples $\leq x$).

Compare empirical vs theoretical to test if data matches a distribution (Kolmogorov-Smirnov test):

In [ ]:
# Generate data from Student-t, fit both Normal and Student-t
true_data = np.random.standard_t(df=5, size=5000)

# Fit normal to data (wrong model)
mu_fit, sigma_fit = true_data.mean(), true_data.std()

# Fit t-distribution to data (correct model)
df_fit, loc_fit, scale_fit = stats.t.fit(true_data)

# Compute empirical CDF
sorted_data = np.sort(true_data)
ecdf = np.arange(1, len(sorted_data) + 1) / len(sorted_data)

# Theoretical CDFs
cdf_normal = stats.norm.cdf(sorted_data, loc=mu_fit, scale=sigma_fit)
cdf_t = stats.t.cdf(sorted_data, df=df_fit, loc=loc_fit, scale=scale_fit)

# KS statistic
ks_normal = np.max(np.abs(ecdf - cdf_normal))
ks_t = np.max(np.abs(ecdf - cdf_t))

# Plot
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Full CDF
ax = axes[0]
ax.plot(sorted_data, ecdf, 'k-', linewidth=2, label='Empirical', alpha=0.7)
ax.plot(sorted_data, cdf_normal, 'r-', linewidth=2, label=f'Normal (KS={ks_normal:.4f})', alpha=0.7)
ax.plot(sorted_data, cdf_t, 'g-', linewidth=2, label=f'Student-t (KS={ks_t:.4f})', alpha=0.7)
ax.set_xlabel('Value')
ax.set_ylabel('CDF')
ax.set_title('Full CDF Comparison')
ax.legend()
ax.grid(alpha=0.3)
ax.set_xlim(-8, 8)

# Q-Q plots
for idx, (model, name) in enumerate([(stats.norm, 'Normal'), (lambda: stats.t(df_fit, loc_fit, scale_fit), 'Student-t')]):
    ax = axes[1] if idx == 1 else plt.subplot(2, 2, 2)
    
    if model == stats.norm:
        theoretical = stats.norm.ppf(ecdf, loc=mu_fit, scale=sigma_fit)
    else:
        theoretical = stats.t.ppf(ecdf, df=df_fit, loc=loc_fit, scale=scale_fit)
    
    # Only plot Q-Q for t-dist in the right subplot
    if idx == 1:
        ax.scatter(theoretical, sorted_data, alpha=0.5, s=20, edgecolors='black', linewidth=0.5)
        ax.plot([-8, 8], [-8, 8], 'k--', linewidth=2, label='Perfect fit')
        ax.set_xlabel('Theoretical Quantiles')
        ax.set_ylabel('Data Quantiles')
        ax.set_title(f'Q-Q Plot: Student-t (KS={ks_t:.4f})')
        ax.legend()
        ax.grid(alpha=0.3)
        ax.set_xlim(-8, 8)
        ax.set_ylim(-8, 8)

plt.tight_layout()
plt.show()

print(f"Kolmogorov-Smirnov Test Results:")
print(f"  Normal:     KS statistic = {ks_normal:.6f}")
print(f"  Student-t:  KS statistic = {ks_t:.6f}")
print(f"\nSmaller KS = better fit. Student-t fits much better (as expected).")
print(f"\nNote: This tests whether empirical ≈ theoretical, not which is 'right'.")

<details>

<summary>What's the difference between CDF and PDF?</summary>
**PDF (Probability Density Function)**: $f(x) = \frac{d}{dx}F(x)$. Area under PDF = probability.<br><br>
**CDF**: $F(x) = \int_{-\infty}^x f(u) du$. Always defined (even for heavy-tailed where variance is infinite).<br><br>
Use CDF for: computing quantiles, tail probabilities, goodness-of-fit tests. Use PDF for: visualizing shape.
</details>

## Real-World Example: Value at Risk (VaR)
Banks use **Value at Risk** to quantify portfolio loss risk. It's literally a quantile!

"95% VaR = worst 5% of days, what's the max loss?" = $F^{-1}(0.05)$ on the left tail.

In [ ]:
# Portfolio returns under different models
portfolio_returns = normal_returns * 100  # Scale to percentage

# Compute VaR (5th percentile = worst 5% of days)
var_95_pct = 5  # percentile

# Different methods
var_empirical = np.percentile(portfolio_returns, var_95_pct)
var_normal = np.mean(portfolio_returns) + np.std(portfolio_returns) * stats.norm.ppf(var_95_pct / 100)
var_t = np.mean(portfolio_returns) + np.std(portfolio_returns) * stats.t.ppf(var_95_pct / 100, df=4)

# Plot
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Histogram with VaR lines
ax = axes[0]
ax.hist(portfolio_returns, bins=50, alpha=0.7, edgecolor='black', density=True)
ax.axvline(var_empirical, color='red', linestyle='--', linewidth=2, label=f'Empirical VaR: {var_empirical:.2f}%')
ax.axvline(var_normal, color='green', linestyle='--', linewidth=2, label=f'Normal VaR: {var_normal:.2f}%')
ax.axvline(var_t, color='orange', linestyle='--', linewidth=2, label=f'Student-t VaR: {var_t:.2f}%')
ax.set_xlabel('Daily Return (%)')
ax.set_ylabel('Density')
ax.set_title('Value at Risk: What does a bad day look like?')
ax.legend()
ax.grid(alpha=0.3)

# Comparison table
ax = axes[1]
ax.axis('off')

risk_data = [
    ['Metric', 'Value'],
    ['Portfolio Daily Return (mean)', f"{np.mean(portfolio_returns):.4f}%"],
    ['Daily Volatility (std dev)', f"{np.std(portfolio_returns):.4f}%"],
    ['', ''],
    ['95% VaR Estimates (Worst 5% of days):', ''],
    ['  Empirical (from data)', f"{var_empirical:.4f}%"],
    ['  Normal model', f"{var_normal:.4f}%"],
    ['  Student-t model', f"{var_t:.4f}%"],
    ['', ''],
    ['Interpretation:', 'In worst 5% of days,'],
    [' ', 'portfolio loses > 2.46% on average.'],
]

table = ax.table(cellText=risk_data, cellLoc='left', loc='center', bbox=[0, 0, 1, 1])
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 2)

# Style header
for i in range(2):
    table[(0, i)].set_facecolor('#40466e')
    table[(0, i)].set_text_props(weight='bold', color='white')

# Style risk estimate rows
for i in range(5, 8):
    table[(i, 0)].set_facecolor('#ffffcc')
    table[(i, 1)].set_facecolor('#ffffcc')

plt.tight_layout()
plt.show()

print(f"Value at Risk Analysis:")
print(f"  Empirical VaR (95%):  {var_empirical:.4f}%")
print(f"  Normal assumption:    {var_normal:.4f}%  (underestimates risk by {abs(var_empirical - var_normal):.4f}%)")
print(f"  Student-t model:      {var_t:.4f}%  (closer to empirical)")
print(f"\nThis is why the 2008 crisis surprised banks: they used Normal models.")
print(f"Heavy tails meant 5-sigma events happened multiple times per year.")

## Summary: Distributions in Practice

| Aspect | Key Point |
|--------|----------|
| **Sampling** | Use inverse transform if CDF invertible; rejection sampling for complex PDFs |
| **Heavy Tails** | Real data (returns, disasters) has fatter tails than Normal; use Student-t or power-law models |
| **CDFs** | Universal tool; use for quantiles, tail probabilities, goodness-of-fit |
| **Risk** | Ignoring heavy tails leads to catastrophic underestimation (2008, anyone?) |

**Rule of thumb**: If you're modeling rare events (losses, crashes, catastrophes), assume heavy tails until proven otherwise.

## Strengths and Weaknesses

### Strengths
- **Universality**: Probability framework applies to all domains (finance, physics, ML, epidemiology)
- **Quantification**: Turns vague "risk" into precise numbers (VaR, CVaR, probability bounds)
- **Predictability**: Under stated assumptions, we can forecast rare events, confidence intervals
- **Composability**: Combine distributions via convolution, mixture models, hierarchical models
- **Statistical foundations**: Hypothesis tests, confidence intervals, maximum likelihood estimation

### Weaknesses
- **Model risk**: Real data rarely matches theoretical distributions perfectly
- **Tail estimation**: Hard to infer tail behavior from finite samples (rare events, by definition, are rare)
- **Dimensionality curse**: High-dimensional distributions are hard to sample and visualize
- **Parameter estimation**: Finite samples introduce uncertainty; rare events dominated by noise
- **Non-stationarity**: Real systems change over time; fitted distributions become obsolete
- **Black swan events**: Completely unprecedented events (by definition) outside any historical distribution